# Week 6 Graded Mini Project: Large Language Models & Embeddings

**Course Outcomes Addressed**
- Demonstrate embedding model applications.
- Implement vector-based retrieval systems.
- Explain semantic similarity concepts.
- Assess vector search performance.

---

## How to use this notebook (Google Colab)
1. Open this notebook in Colab.
2. Run cells top-to-bottom.
3. Replace placeholder values where asked.
4. Keep all outputs visible for submission evidence.

---

## IMPORTANT: HF Token Setup (Environment Variable)
You requested environment-variable based setup with `HF_TOKEN`.

### Option A (Quick, session-only)
Set directly in a code cell:
```python
import os
os.environ["HF_TOKEN"] = "hf_xxxxxxxxxxxxxxxxx"
```

### Option B (Recommended in Colab: Secrets)
- In Colab, open **Secrets** (left panel, key icon).
- Add secret name: `HF_TOKEN`
- Add your token value.
- Use this in code:
```python
from google.colab import userdata
import os
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
```

> ⚠️ Never commit your token into GitHub. Keep it in Colab secrets or local env only.


## 0) Install & Imports

In [ ]:
!pip -q install transformers gensim scikit-learn seaborn matplotlib

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from transformers import pipeline, AutoTokenizer
import gensim.downloader as api
from sklearn.metrics.pairwise import cosine_similarity

## 1) HF Token via Environment Variable (`HF_TOKEN`)

In [ ]:
# OPTION A: Quick direct assignment (session only)
# Replace with your real token and run once per session
# os.environ["HF_TOKEN"] = "hf_xxxxxxxxxxxxxxxxx"

# OPTION B (Recommended in Colab): read from Colab Secrets
# from google.colab import userdata
# os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

token_present = bool(os.environ.get("HF_TOKEN"))
print("HF_TOKEN found in environment:", token_present)

---
# Section A: LLM Foundations & Hugging Face

## A1) Hugging Face Setup & Text Generation
Generate **three different continuations** for:
`AI is transforming industries by ...`

In [ ]:
generator = pipeline(
    task="text-generation",
    model="distilgpt2"
)

prompt = "AI is transforming industries by"
gen_outputs = generator(
    prompt,
    max_length=55,
    num_return_sequences=3,
    do_sample=True,
    temperature=0.9,
    top_k=50
)

for i, out in enumerate(gen_outputs, 1):
    print(f"\nContinuation {i}:\n{out['generated_text']}")

## A2) Tokenisation Demo
Sentence: `LLMs are powerful tools for natural language understanding.`

Display:
- Tokens
- Token IDs
- Sequence length

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilgpt2")

sentence = "LLMs are powerful tools for natural language understanding."
tokens = tokenizer.tokenize(sentence)
token_ids = tokenizer.encode(sentence)

print("Tokens:")
print(tokens)
print("\nToken IDs:")
print(token_ids)
print("\nSequence Length:", len(token_ids))

---
# Section B: Prompt Engineering

## B1) Prompt Tuning Challenge
Design and run 3 prompts:
1. Summarisation (≤ 30 words)
2. Q&A
3. Creative generation (4-line AI poem)

In [ ]:
prompts = {
    "Summarisation": "Summarise in 30 words or fewer: Artificial intelligence automates repetitive tasks, supports better decisions from data, and improves customer service with chatbots and recommendation systems.",
    "Q&A": "Question: What is the capital of Japan?\nAnswer:",
    "Creative": "Write a 4-line poem about AI and the future:"
}

outputs_b = {}
for task, p in prompts.items():
    result = generator(
        p,
        max_length=100,
        num_return_sequences=1,
        do_sample=True,
        temperature=0.8
    )[0]["generated_text"]
    outputs_b[task] = result

for task, text in outputs_b.items():
    print(f"\n--- {task} ---")
    print(text)

## B2) Reflection (100–150 words)
You may adapt this sample reflection in your own words:

Rephrasing prompts changed the output quality, structure, and relevance significantly. When I added explicit constraints such as ‘30 words or fewer,’ the model attempted concise responses and focused on essential details. In Q&A format, adding clear labels like ‘Question’ and ‘Answer’ guided the model toward direct factual completion rather than open-ended continuation. For creative generation, broad and imaginative phrasing produced more varied and expressive text, but also less predictability. I observed that prompt clarity improves alignment with task goals, while vague prompts increase randomness and off-topic outputs. Overall, prompt engineering acts as a practical control mechanism for LLM behavior without retraining, especially for formatting, tone, and task-specific intent in real-world workflows.

---
# Section C: Embeddings with Gensim

## C1) Load GloVe Embeddings
Model: `glove-wiki-gigaword-50`

In [ ]:
glove = api.load("glove-wiki-gigaword-50")
print("Vector size:", glove.vector_size)

## C2) Word Embeddings Analysis
Use words: `king`, `queen`, `diamond`

For each word:
- First 10 vector values
- Top 5 most similar words + similarity scores

In [ ]:
words = ["king", "queen", "diamond"]

for w in words:
    print(f"\nWord: {w}")
    print("First 10 vector values:")
    print(glove[w][:10])

    print("Top 5 most similar words:")
    for similar_word, score in glove.most_similar(w, topn=5):
        print(f"  {similar_word}: {score:.4f}")

## C3) Sentence-Level Embeddings & Similarity Matrix
Create 5 short sentences, represent each by mean word vectors, then compute cosine similarity matrix.

In [ ]:
sentences = [
    "AI improves customer support with chatbots.",
    "Machine learning helps detect fraud in banking.",
    "Neural networks assist medical image diagnosis.",
    "Diamonds are precious stones used in jewellery.",
    "Gold and silver are common jewellery materials."
]

def sentence_embedding(sent, model):
    tokens = [t.lower().strip(".,!?;:") for t in sent.split()]
    vecs = [model[t] for t in tokens if t in model]
    if len(vecs) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vecs, axis=0)

embeddings = np.array([sentence_embedding(s, glove) for s in sentences])
sim_mat = cosine_similarity(embeddings)

sim_df = pd.DataFrame(sim_mat, index=sentences, columns=sentences)
sim_df

In [ ]:
plt.figure(figsize=(10, 6))
sns.heatmap(sim_df, annot=True, cmap="YlGnBu", fmt=".2f")
plt.title("Sentence Cosine Similarity Matrix")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

---
# Section D: Application Exploration

## D1) Transformer Pipeline Demo (Sentiment Classification)
(You may switch to translation or summarization if preferred.)

In [ ]:
sentiment_model = pipeline("sentiment-analysis")

custom_input = "The AI assistant reduced our support response time by 40%, and customer satisfaction has improved."
sentiment_result = sentiment_model(custom_input)

print("Input:", custom_input)
print("Output:", sentiment_result)

## D2) Reflection (~100 words)
Sample reflection (edit in your own style):

Sentiment classification can be valuable in business workflows that process large volumes of customer text, such as reviews, survey comments, and support tickets. Instead of manually reading each entry, teams can automatically label messages as positive, negative, or neutral and monitor changes over time. This helps identify service issues early, prioritize critical complaints, and measure the impact of product improvements. In professional settings, sentiment dashboards can support customer success, marketing, and operations teams by converting unstructured language into actionable indicators. Overall, transformer-based sentiment analysis improves speed, consistency, and scalability of feedback analysis in data-driven organizations.

---
# Final Submission Checklist

## Notebook (required)
- [ ] Section A outputs included (generation + tokenization)
- [ ] Section B outputs + 100–150 word reflection
- [ ] Section C outputs (word vectors, similar words, similarity matrix)
- [ ] Section D pipeline demo + ~100 word reflection

## PDF Summary (required)
Include:
1. Screenshots of key outputs
2. What you learned from each section
3. Key observations: LLMs vs Embeddings
4. Real-world applications of vector search & embeddings

## Suggested naming
`Week 6_Graded Project_[Your Name].zip` or separate notebook + PDF uploads.